# খরচ দাবি বিশ্লেষণ

এই নোটবুকটি প্রদর্শন করে কীভাবে এমন এজেন্ট তৈরি করা যায় যারা প্লাগইন ব্যবহার করে স্থানীয় রসিদ ইমেজ থেকে ভ্রমণ খরচ প্রক্রিয়া করে, একটি খরচ দাবি ইমেল তৈরি করে, এবং একটি পাই চার্ট ব্যবহার করে খরচের ডেটা ভিজ্যুয়ালাইজ করে। এজেন্টরা কাজের প্রসঙ্গ অনুযায়ী ফাংশন নির্বাচন করে।

ধাপসমূহ:
1. OCR এজেন্ট স্থানীয় রসিদ ইমেজ প্রক্রিয়া করে এবং ভ্রমণ খরচের তথ্য নিষ্কাশন করে।
2. ইমেল এজেন্ট একটি খরচ দাবি ইমেল তৈরি করে।

### ভ্রমণ খরচ পরিস্থিতির উদাহরণ:
ভাবুন আপনি একজন কর্মচারী যিনি অন্য একটি শহরে ব্যবসায়িক বৈঠকের জন্য সফর করছেন। আপনার কোম্পানির একটি নীতি রয়েছে যা সমস্ত যুক্তিসঙ্গত ভ্রমণ-সংশ্লিষ্ট খরচ ফেরত দেওয়ার। এখানে সম্ভাব্য ভ্রমণ খরচের একটি ভাঙ্গন:
- পরিবহন:
আপনার বাড়ির শহর থেকে গন্তব্য শহরে ট্রিপের জন্য এয়ারফেয়ার।
এয়ারপোর্টে যাতায়াতের জন্য ট্যাক্সি বা রাইড-হেলিং সেবা।
গন্তব্য শহরে স্থানীয় পরিবহন (যেমন পাবলিক ট্রানজিট, ভাড়া কার, অথবা ট্যাক্সি)।

- থাকার ব্যবস্থা:
ব্যবসায়িক বৈঠক স্থলের কাছাকাছি মধ্যমূল্যের ব্যবসায়িক হোটেলে তিন রাতের থাকার ব্যবস্থা।

- খাবার:
প্রতিদিনের খাদ্য ভাতা, সকালের নাস্তা, মধ্যাহ্নভোজ এবং রাত্রিভোজের জন্য, কোম্পানির পের ডিম নীতিমতো।

- বিবিধ খরচ:
এয়ারপোর্টে পার্কিং ফি।
হোটেলে ইন্টারনেট ব্যবহারের চার্জ।
টিপ অথবা ছোট সার্ভিস চার্জ।

- ডকুমেন্টেশন:
আপনি সমস্ত রসিদ (ফ্লাইট, ট্যাক্সি, হোটেল, খাবার ইত্যাদি) এবং একটি পূর্ণ খরচ রিপোর্ট রিফান্ডের জন্য জমা দেন।


## Import required libraries

নোটবুকের জন্য প্রয়োজনীয় লাইব্রেরি এবং মডিউলগুলি আমদানি করুন।


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## ব্যয় মডেল সংজ্ঞায়িত করুন

 পৃথক ব্যয়গুলির জন্য একটি Pydantic মডেল এবং ব্যবহারকারীর প্রশ্নকে গঠিত ব্যয় ডেটায় রূপান্তর করার জন্য একটি ExpenseFormatter ক্লাস তৈরি করুন।

 প্রতিটি ব্যয় নিম্নলিখিত ফর্ম্যাটে উপস্থাপিত হবে:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## টুলস সংজ্ঞায়ন - ইমেইল তৈরি করা

একটি টুল ফাংশন তৈরি করুন যা একটি খরচ দাবি পাঠানোর জন্য একটি ইমেইল তৈরি করে।  
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।  
- এটি খরচের মোট পরিমাণ হিসাব করে এবং বিস্তারিত একটি ইমেইল বডিতে ফরম্যাট করে।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# রসিদ ইমেজ থেকে ভ্রমণ খরচ বের করার টুল

রসিদ ইমেজ থেকে ভ্রমণ খরচ বের করার জন্য একটি টুল ফাংশন তৈরি করুন।
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।
- এটি রসিদ ইমেজটি পড়ে, সেটিকে base64 এ এনকোড করে, এবং এজেন্টের বিশ্লেষণের জন্য ডেটা URI প্রদান করে।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

এজেন্টগুলি সংজ্ঞায়িত করুন এবং তাদের একটি ধারাবাহিক ওয়ার্কফ্লোতে সংযুক্ত করুন `WorkflowBuilder` ব্যবহার করে।
- OCR এজেন্টটি `load_receipt_image` টুল ব্যবহার করে রশিদ ছবিটি থেকে কাঠামোবদ্ধ খরচের তথ্য আহরণ করে।
- ইমেইল এজেন্টটি আহরিত তথ্য গ্রহণ করে এবং `generate_expense_email` টুল ব্যবহার করে একটি পেশাদার খরচ দাবি ইমেইল তৈরি করে।
- `WorkflowBuilder` এর `add_edge` দিয়ে একটি ধারাবাহিক পাইপলাইন তৈরি করে: OCR এজেন্ট → ইমেইল এজেন্ট।


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

ক্রমিক ওয়ার্কফ্লো তৈরি করুন এবং রসিদ চিত্রটি প্রক্রিয়া করতে এবং ব্যয় দাবির ইমেল তৈরি করতে এটি চালান।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
